# TEME Survival V3 - Colab Ready (Verified)

This notebook is designed to run in Google Colab and keeps TEME runtime compatibility.

What this notebook guarantees:
- Uses the same 5-feature TEME runtime contract: growth_rate_class, drought_score, fire_score, disease_score, planted_count.
- Applies strict cleaning: type coercion, missing handling, duplicate handling, range clipping.
- Builds context enrichment (climate and fire proxies) before training.
- Trains compact model candidates and selects best by MAE.
- Saves reproducible artifacts + optional runtime-compatible model path.

Run cells in order from top to bottom.

In [ ]:
import json
import random
import datetime as dt
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Colab execution settings
USE_GOOGLE_DRIVE = True
INSTALL_MISSING_PACKAGES = True
REPO_ROOT = Path('/content/carbonsense') if Path('/content/carbonsense').exists() else Path('/content')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content')

if INSTALL_MISSING_PACKAGES:
    for pkg in ['xarray', 'netCDF4', 'openpyxl']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

RUN_ID = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR = DRIVE_ROOT / 'TEME_ML_checkpoints' / f'teme_survival_v3_verified_{RUN_ID}'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BASE_DATASET_CANDIDATES = [
    REPO_ROOT / 'data' / 'datasets' / 'v_survival_training.csv',
    DRIVE_ROOT / 'v_survival_training.csv',
    Path('/content/v_survival_training.csv'),
]
BASE_DATASET_PATH = next((p for p in BASE_DATASET_CANDIDATES if p.exists()), BASE_DATASET_CANDIDATES[0])

RAW_ANCHOR_CANDIDATES = [
    DRIVE_ROOT / 'TEME_data' / 'real_anchor_raw.csv',
    REPO_ROOT / 'real_anchor_raw.csv',
    Path('/content/real_anchor_raw.csv'),
]
RAW_ANCHOR_PATH = next((p for p in RAW_ANCHOR_CANDIDATES if p.exists()), RAW_ANCHOR_CANDIDATES[0])
CLEAN_ANCHOR_PATH = ARTIFACT_DIR / 'real_anchor_survival_clean.csv'
HYBRID_OUTPUT_PATH = ARTIFACT_DIR / 'v_survival_training_v3_hybrid.csv'

TERRACLIMATE_URL_BASE = 'https://climate.northwestknowledge.net/TERRACLIMATE-DATA'
GFW_FIRE_URL = 'https://gfw2-data.s3.amazonaws.com/country-pages/country_stats/download/2024/IND.xlsx'

def ckpt(name: str, payload: dict | None = None):
    stamp = dt.datetime.now().strftime('%H:%M:%S')
    print(f'[CHECKPOINT {stamp}] {name}')
    if payload is not None:
        with (ARTIFACT_DIR / f'{name}.json').open('w', encoding='utf-8') as f:
            json.dump(payload, f, indent=2, default=str)

print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('BASE_DATASET_PATH:', BASE_DATASET_PATH)
print('RAW_ANCHOR_PATH:', RAW_ANCHOR_PATH)
print('NOTE: If RAW_ANCHOR_PATH does not exist, training will proceed using base synthetic data only.')

In [ ]:
def safe_download(url: str, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f'Downloading: {url}')
        urlretrieve(url, target)
    else:
        print(f'Using cached file: {target.name}')

def build_pdsi_monthly_features(year_start=2010, year_end=2025) -> pd.DataFrame:
    import xarray as xr

    tc_root = DRIVE_ROOT / 'TEME_data' / 'terraclimate' / 'pdsi_india_2010_2025'
    global_dir = tc_root / 'global_nc'
    india_dir = tc_root / 'india_subset_nc'
    global_dir.mkdir(parents=True, exist_ok=True)
    india_dir.mkdir(parents=True, exist_ok=True)

    lat_min, lat_max = 6.0, 38.0
    lon_min, lon_max = 68.0, 98.0

    for yr in range(year_start, year_end + 1):
        src = f'{TERRACLIMATE_URL_BASE}/TerraClimate_pdsi_{yr}.nc'
        gfp = global_dir / f'TerraClimate_pdsi_{yr}.nc'
        ifp = india_dir / f'TerraClimate_pdsi_india_{yr}.nc'
        safe_download(src, gfp)

        if not ifp.exists():
            ds = xr.open_dataset(gfp)
            if float(ds['lon'].max()) > 180.0:
                ds = ds.assign_coords(lon=(((ds['lon'] + 180) % 360) - 180)).sortby('lon')
            lat_slice = slice(lat_max, lat_min) if ds['lat'][0] > ds['lat'][-1] else slice(lat_min, lat_max)
            india = ds.sel(lat=lat_slice, lon=slice(lon_min, lon_max))
            india.to_netcdf(ifp)
            ds.close()
            india.close()

    rows = []
    for fp in sorted(india_dir.glob('TerraClimate_pdsi_india_*.nc')):
        ds = xr.open_dataset(fp)
        times = pd.to_datetime(ds['time'].values)
        p = ds['pdsi']
        for i, t in enumerate(times):
            arr = p.isel(time=i).values.astype(float)
            arr = arr[np.isfinite(arr)]
            if arr.size == 0:
                continue
            rows.append({
                'year': int(t.year),
                'month': int(t.month),
                'pdsi_mean': float(np.mean(arr)),
                'pdsi_std': float(np.std(arr)),
                'pdsi_drought_share_le_m2': float(np.mean(arr <= -2.0))
            })
        ds.close()

    cdf = pd.DataFrame(rows)
    if cdf.empty:
        return pd.DataFrame(columns=['year', 'month', 'pdsi_mean', 'pdsi_std', 'pdsi_drought_share_le_m2'])
    return cdf.sort_values(['year', 'month']).drop_duplicates(['year', 'month'], keep='last').reset_index(drop=True)

def build_fire_features() -> pd.DataFrame:
    fire_root = DRIVE_ROOT / 'TEME_data' / 'fire' / 'gfw_india_2024'
    fire_root.mkdir(parents=True, exist_ok=True)
    xlsx = fire_root / 'IND.xlsx'
    safe_download(GFW_FIRE_URL, xlsx)

    xls = pd.ExcelFile(xlsx)
    fire_tokens = ['fire', 'burn', 'burned', 'alert', 'hotspot']
    rows = []

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(xlsx, sheet_name=sheet)
        except Exception:
            continue
        if df.empty:
            continue

        cols = {c: str(c).strip().lower() for c in df.columns}
        year_col = next((c for c, lc in cols.items() if lc in ['year', 'yr']), None)
        if year_col is None:
            df['year'] = 2024
            year_col = 'year'
        region_col = next((c for c, lc in cols.items() if 'state' in lc or 'province' in lc), None)
        if region_col is None:
            df['region'] = 'INDIA_ALL'
            region_col = 'region'

        for c, lc in cols.items():
            if any(tok in lc for tok in fire_tokens):
                v = pd.to_numeric(df[c], errors='coerce')
                m = v.notna()
                if m.any():
                    rows.append(pd.DataFrame({
                        'region': df.loc[m, region_col].astype(str).str.strip().replace('', 'INDIA_ALL'),
                        'year': pd.to_numeric(df.loc[m, year_col], errors='coerce').fillna(2024).astype(int),
                        'feature_name': lc,
                        'value': v.loc[m].astype(float)
                    }))

    if not rows:
        return pd.DataFrame({'region': ['INDIA_ALL'], 'year': [2024], 'fire_score_proxy_1_10': [5.0]})

    long_df = pd.concat(rows, ignore_index=True)
    agg = (
        long_df.groupby(['region', 'year', 'feature_name'], as_index=False)['value']
        .mean()
        .pivot_table(index=['region', 'year'], columns='feature_name', values='value', aggfunc='mean')
        .reset_index()
    )
    agg.columns = [str(c) for c in agg.columns]

    value_cols = [c for c in agg.columns if c not in ['region', 'year']]
    z = agg[value_cols].copy()
    z = (z - z.mean()) / z.std(ddof=0).replace(0, np.nan)
    z = z.fillna(0.0)
    q = z.mean(axis=1).rank(pct=True)
    agg['fire_score_proxy_1_10'] = (1 + 9 * q).round(2)
    return agg[['region', 'year', 'fire_score_proxy_1_10']]

def clean_anchor(raw_path: Path) -> pd.DataFrame:
    if not raw_path.exists():
        raise FileNotFoundError(f'Raw anchor file not found: {raw_path}')

    raw = pd.read_csv(raw_path)
    raw.columns = [str(c).strip() for c in raw.columns]

    numeric_cols = ['Total Plant No', 'Plant Survived', 'Total Area Covered', 'Plant height']
    for c in numeric_cols:
        if c in raw.columns:
            raw[c] = pd.to_numeric(raw[c], errors='coerce')

    raw['observation_date'] = pd.to_datetime(raw.get('Registered On'), dayfirst=True, errors='coerce')

    keep = (
        raw['Total Plant No'].gt(0)
        & raw['Plant Survived'].ge(0)
        & raw['Plant Survived'].le(raw['Total Plant No'])
        & raw['observation_date'].notna()
    )
    a = raw.loc[keep].copy()

    a['planted_count'] = a['Total Plant No'].round().astype(int)
    a['actual_alive_count'] = a['Plant Survived'].round().astype(int)
    a['actual_dead_count'] = (a['planted_count'] - a['actual_alive_count']).clip(lower=0).astype(int)
    a['survival_ratio'] = (a['actual_alive_count'] / a['planted_count']).clip(0.0, 1.0)

    a['region'] = a.get('State', pd.Series(index=a.index)).fillna('INDIA_ALL').astype(str).str.strip().replace('', 'INDIA_ALL')
    a['species_name'] = 'UNKNOWN_NEEDS_ENRICHMENT'
    a['row_origin'] = 'real'
    a['source_id'] = 'egreenwatch_recent_progress'

    dedupe_cols = ['State', 'Division', 'Range', 'Year', 'Stage', 'planted_count', 'actual_alive_count']
    existing_dedupe = [c for c in dedupe_cols if c in a.columns]
    if existing_dedupe:
        a = a.sort_values('observation_date').drop_duplicates(subset=existing_dedupe, keep='last')
    a = a.drop_duplicates(subset=['observation_date', 'region', 'planted_count', 'actual_alive_count'], keep='last')

    if 'Plant height' not in a.columns:
        a['Plant height'] = np.nan
    if 'Total Area Covered' not in a.columns:
        a['Total Area Covered'] = np.nan
    a['Plant height'] = a['Plant height'].fillna(a['Plant height'].median())
    a['Total Area Covered'] = a['Total Area Covered'].fillna(a['Total Area Covered'].median())

    return a.reset_index(drop=True)

def enrich_and_preprocess(anchor_df: pd.DataFrame, climate_df: pd.DataFrame, fire_df: pd.DataFrame) -> pd.DataFrame:
    d = anchor_df.copy()
    d['year'] = d['observation_date'].dt.year.astype(int)
    d['month'] = d['observation_date'].dt.month.astype(int)

    d = d.merge(climate_df, how='left', on=['year', 'month'])
    d = d.merge(fire_df[['region', 'year', 'fire_score_proxy_1_10']], how='left', on=['region', 'year'])

    for c in ['pdsi_mean', 'pdsi_std', 'pdsi_drought_share_le_m2', 'fire_score_proxy_1_10']:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce').fillna(d[c].median())

    drought_rank = d['pdsi_mean'].rank(pct=True, method='average')
    d['drought_score'] = (10 - 9 * drought_rank).clip(1, 10).round(2)
    d['fire_score'] = d['fire_score_proxy_1_10'].clip(1, 10).round(2)

    d['growth_rate_class'] = 6.0
    d['disease_score'] = 5.0

    d = d.drop_duplicates(subset=['observation_date', 'region', 'planted_count', 'actual_alive_count'])
    req = ['growth_rate_class', 'drought_score', 'fire_score', 'disease_score', 'planted_count', 'survival_ratio']
    for c in req:
        d[c] = pd.to_numeric(d[c], errors='coerce').fillna(d[c].median())

    d['survival_ratio'] = d['survival_ratio'].clip(0.0, 1.0)
    d['growth_rate_class'] = d['growth_rate_class'].clip(1, 10)
    d['drought_score'] = d['drought_score'].clip(1, 10)
    d['fire_score'] = d['fire_score'].clip(1, 10)
    d['disease_score'] = d['disease_score'].clip(1, 10)
    d['planted_count'] = d['planted_count'].clip(lower=1)

    return d.reset_index(drop=True)

def preflight_schema_assertions(df: pd.DataFrame):
    required = ['growth_rate_class', 'drought_score', 'fire_score', 'disease_score', 'planted_count', 'survival_ratio']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required training columns: {missing}')

    if df[required].isna().sum().sum() > 0:
        raise ValueError('Required columns still contain nulls after preprocessing.')

    if not ((df['survival_ratio'] >= 0.0) & (df['survival_ratio'] <= 1.0)).all():
        raise ValueError('survival_ratio out of bounds [0,1].')

    if not ((df['growth_rate_class'] >= 1) & (df['growth_rate_class'] <= 10)).all():
        raise ValueError('growth_rate_class out of bounds [1,10].')

    if not ((df['drought_score'] >= 1) & (df['drought_score'] <= 10)).all():
        raise ValueError('drought_score out of bounds [1,10].')

    if not ((df['fire_score'] >= 1) & (df['fire_score'] <= 10)).all():
        raise ValueError('fire_score out of bounds [1,10].')

    if not ((df['disease_score'] >= 1) & (df['disease_score'] <= 10)).all():
        raise ValueError('disease_score out of bounds [1,10].')

    if not (df['planted_count'] > 0).all():
        raise ValueError('planted_count must be > 0.')

In [ ]:
ckpt('01_start')

if not BASE_DATASET_PATH.exists():
    raise FileNotFoundError(f'Base dataset not found: {BASE_DATASET_PATH}')

climate_df = build_pdsi_monthly_features(2010, 2025)
fire_df = build_fire_features()
ckpt('02_context_features_ready', {
    'climate_shape': tuple(climate_df.shape),
    'fire_shape': tuple(fire_df.shape),
})

if RAW_ANCHOR_PATH.exists():
    anchor_clean = clean_anchor(RAW_ANCHOR_PATH)
    enriched_real = enrich_and_preprocess(anchor_clean, climate_df, fire_df)
    enriched_real.to_csv(CLEAN_ANCHOR_PATH, index=False)
    ckpt('03_anchor_clean_enriched', {
        'anchor_clean_shape': tuple(anchor_clean.shape),
        'enriched_shape': tuple(enriched_real.shape),
        'clean_anchor_path': str(CLEAN_ANCHOR_PATH),
    })
else:
    print(f'Anchor file missing: {RAW_ANCHOR_PATH}. Proceeding with base synthetic data only.')
    enriched_real = pd.DataFrame()

base_df = pd.read_csv(BASE_DATASET_PATH)
base_df['row_origin'] = base_df.get('row_origin', 'synthetic')
base_df['source_id'] = base_df.get('source_id', 'base_canonical')

required = ['growth_rate_class', 'drought_score', 'fire_score', 'disease_score', 'planted_count', 'survival_ratio']
for c in required:
    if c not in base_df.columns:
        base_df[c] = np.nan
    base_df[c] = pd.to_numeric(base_df[c], errors='coerce')

if not enriched_real.empty:
    real_for_hybrid = enriched_real.copy()
    real_for_hybrid['row_origin'] = 'real'
    real_for_hybrid['source_id'] = 'egreenwatch_plus_context'
    hybrid = pd.concat([base_df, real_for_hybrid], ignore_index=True, sort=False).drop_duplicates()
else:
    hybrid = base_df.copy()
for c in required:
    hybrid[c] = pd.to_numeric(hybrid[c], errors='coerce').fillna(hybrid[c].median())

hybrid['survival_ratio'] = hybrid['survival_ratio'].clip(0.0, 1.0)
hybrid['growth_rate_class'] = hybrid['growth_rate_class'].clip(1, 10)
hybrid['drought_score'] = hybrid['drought_score'].clip(1, 10)
hybrid['fire_score'] = hybrid['fire_score'].clip(1, 10)
hybrid['disease_score'] = hybrid['disease_score'].clip(1, 10)
hybrid['planted_count'] = hybrid['planted_count'].clip(lower=1)

hybrid['species_name'] = hybrid.get('species_name', 'unknown').fillna('unknown').astype(str)
hybrid['region'] = hybrid.get('region', 'unknown').fillna('unknown').astype(str)
hybrid['dataset_version'] = 'teme_survival_v3_verified_colab'

preflight_schema_assertions(hybrid)
hybrid.to_csv(HYBRID_OUTPUT_PATH, index=False)

print('=== Hybrid preview (head 10) ===')
display(hybrid.head(10))
print('=== Hybrid shape ===', hybrid.shape)
print('=== row_origin mix ===')
print(hybrid['row_origin'].value_counts(dropna=False))
print('=== Nulls in required columns ===')
print(hybrid[required].isna().sum())
ckpt('04_hybrid_ready', {'hybrid_shape': tuple(hybrid.shape), 'hybrid_csv': str(HYBRID_OUTPUT_PATH)})

In [ ]:
X_cols = ['growth_rate_class', 'drought_score', 'fire_score', 'disease_score', 'planted_count']
y_col = 'survival_ratio'
groups = hybrid['species_name'] + '::' + hybrid['region']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(splitter.split(hybrid, groups=groups))
train_df = hybrid.iloc[tr_idx].copy()
test_df = hybrid.iloc[te_idx].copy()

X_train = train_df[X_cols].copy()
y_train = train_df[y_col].copy()
X_test = test_df[X_cols].copy()
y_test = test_df[y_col].copy()

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), X_cols),
])

models = {
    'random_forest': RandomForestRegressor(n_estimators=350, max_depth=14, min_samples_leaf=2, random_state=SEED, n_jobs=1),
    'extra_trees': ExtraTreesRegressor(n_estimators=400, max_depth=16, min_samples_leaf=2, random_state=SEED, n_jobs=1),
}

results = []
best = None
for name, reg in models.items():
    pipe = Pipeline([('pre', pre), ('model', reg)])
    pipe.fit(X_train, y_train)
    pred = np.clip(pipe.predict(X_test), 0.0, 1.0)
    res = {
        'name': name,
        'mae': float(mean_absolute_error(y_test, pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
        'r2': float(r2_score(y_test, pred)),
        'model': pipe,
    }
    results.append(res)
    if best is None or res['mae'] < best['mae']:
        best = res

metrics = {
    'run_id': RUN_ID,
    'selected_model': best['name'],
    'test_mae': best['mae'],
    'test_rmse': best['rmse'],
    'test_r2': best['r2'],
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'hybrid_shape': tuple(hybrid.shape),
    'base_runtime_features': X_cols,
}

model_path = ARTIFACT_DIR / 'teme_survival_latest.joblib'
metrics_path = ARTIFACT_DIR / 'metrics.json'
joblib.dump(best['model'], model_path)
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

runtime_model_path = REPO_ROOT / 'packages' / 'ml_services' / 'teme' / 'ml' / 'models' / 'rf_survival_v1_2.pkl'
runtime_model_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(model_path, runtime_model_path)

print('=== Model comparison ===')
display(pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in results]).sort_values('mae'))
print('=== Selected metrics ===')
print(json.dumps(metrics, indent=2))
print('Model saved:', model_path)
print('Metrics saved:', metrics_path)
print('Runtime-compatible copy:', runtime_model_path)
ckpt('05_training_done', metrics)